In [ ]:
# ======================================================
# Notebook: 8D Hyperparameter Optimisation (Neural Network surrogate)
# Inputs: (40,8) | Output: (40,)
# Goal: maximise validation score
# ======================================================

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim

# Load data
X = np.load("/mnt/data/initial_inputs.npy")      # (40,8)
y = np.load("/mnt/data/initial_outputs.npy")     # (40,)

# Normalize targets
y_norm = (y - y.mean()) / (y.std() + 1e-8)

X_t = torch.tensor(X, dtype=torch.float32)
y_t = torch.tensor(y_norm.reshape(-1,1), dtype=torch.float32)

# Neural network surrogate
model = nn.Sequential(
    nn.Linear(8, 64),
    nn.ReLU(),
    nn.Dropout(0.2),
    nn.Linear(64, 64),
    nn.ReLU(),
    nn.Dropout(0.2),
    nn.Linear(64, 1)
)

criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

# Train
model.train()
for _ in range(1000):
    optimizer.zero_grad()
    pred = model(X_t)
    loss = criterion(pred, y_t)
    loss.backward()
    optimizer.step()

# Candidate sampling
bounds = [(X[:,i].min(), X[:,i].max()) for i in range(8)]
n_candidates = 10000
X_grid = np.column_stack([
    np.random.uniform(b[0], b[1], n_candidates) for b in bounds
])

X_grid_t = torch.tensor(X_grid, dtype=torch.float32)

# MC Dropout
model.train()
samples = []

with torch.no_grad():
    for _ in range(40):
        samples.append(model(X_grid_t).numpy())

samples = np.stack(samples)
mean_pred = samples.mean(axis=0).flatten()
uncertainty = samples.std(axis=0).flatten()

# Acquisition
acquisition = mean_pred + 0.5 * uncertainty

# Select next (10,8)
top_idx = np.argsort(acquisition)[-10:]
next_points = X_grid[top_idx]

print("Next (10,8) hyperparameter candidates:")
print(next_points)